In [0]:
read_path="s3://olist-batch-processing-project-bucket/silver"
write_path="s3://olist-batch-processing-project-bucket/gold"

customers_read_path = f"{read_path}/customers_data"
geolocation_read_path = f"{read_path}/geolocation_data"
order_items_read_path = f"{read_path}/order_items_data"
order_payments_read_path = f"{read_path}/order_payments_data"
order_reviews_read_path = f"{read_path}/order_reviews_data"
orders_read_path = f"{read_path}/orders_data"
product_category_name_translation_read_path = f"{read_path}/product_category_name_translation_data"
products_read_path = f"{read_path}/products_data"
sellers_read_path = f"{read_path}/sellers_data"



customer_dimension_write_path= f"{write_path}/customer_dimension"
seller_dimension_write_path= f"{write_path}/seller_dimension"
product_dimension_write_path= f"{write_path}/product_dimension"
order_status_dimension_write_path= f"{write_path}/order_status_dimension"
payment_type_dimension_write_path= f"{write_path}/payment_type_dimension"
date_dimension_write_path= f"{write_path}/date_dimension"
review_dimension_write_path= f"{write_path}/review_dimension"
review_fact_write_path= f"{write_path}/review_fact"
order_item_fact_write_path= f"{write_path}/order_item_fact"
order_fact_write_path= f"{write_path}/order_fact"
payment_fact_write_path= f"{write_path}/payment_fact"


In [0]:
df_customers=spark.read.parquet(customers_read_path)
df_geolocation=spark.read.parquet(geolocation_read_path)
df_order_items=spark.read.parquet(order_items_read_path)
df_order_payments=spark.read.parquet(order_payments_read_path)
df_order_reviews=spark.read.parquet(order_reviews_read_path)
df_orders=spark.read.parquet(orders_read_path)
df_product_category_name_translation=spark.read.parquet(product_category_name_translation_read_path)
df_products=spark.read.parquet(products_read_path)
df_sellers=spark.read.parquet(sellers_read_path)

In [0]:
import pyspark.sql.functions as sf

In [0]:
df_customers_dimension=df_customers.join(
                            df_geolocation,how="left",on=df_geolocation.geolocation_zip_code_prefix==df_customers.customer_zip_code_prefix).select(
                                df_customers.customer_unique_id.alias("customer_unique_id"),
                                df_geolocation.geolocation_city.alias("customer_city"),
                                df_geolocation.geolocation_state.alias("customer_state"),
                                df_customers.customer_zip_code_prefix.alias("customer_zip_code_prefix"),
                                df_geolocation.geolocation_lat.alias("customer_estimated_lat"),
                                df_geolocation.geolocation_lng.alias("customer_estimated_lng")
                            )


df_customers_dimension = df_customers_dimension.fillna({
    "customer_city": "UNKNOWN",
    "customer_state": "UKN",
    "customer_estimated_lat": 0.0,
    "customer_estimated_lng": 0.0
})

In [0]:
df_sellers_dimension= df_sellers.join(
    df_geolocation,how="left",on=df_geolocation.geolocation_zip_code_prefix==df_sellers.seller_zip_code_prefix).select(
        df_sellers.seller_id.alias("seller_id"),
        df_sellers.seller_zip_code_prefix.alias("seller_zip_code_prefix"),
        df_geolocation.geolocation_city.alias("seller_city"),
        df_geolocation.geolocation_state.alias("seller_state"),
        df_geolocation.geolocation_lat.alias("seller_estimated_lat"),
        df_geolocation.geolocation_lng.alias("seller_estimated_lng")
)
    
df_sellers_dimension = df_sellers_dimension.fillna({
    "seller_city": "UNKNOWN",
    "seller_state": "UKN",
    "seller_estimated_lat": 0.0,
    "seller_estimated_lng": 0.0
})

In [0]:
df_products_dimension=df_products.join(
    df_product_category_name_translation,how="left",on=df_products.product_category_name==df_product_category_name_translation.product_category_name).select(
        df_products.product_id.alias("product_id"),
        df_products.product_category_name.alias("product_category_name"),
        df_product_category_name_translation.product_category_name_english.alias("product_category_name_english"),
        df_products.product_name_lenght.alias("product_name_length"),
        df_products.product_description_lenght.alias("product_description_length"),
        df_products.product_photos_qty.alias("product_photos_qty"),
        df_products.product_weight_g.alias("product_weight_g"),
        df_products.product_length_cm.alias("product_length_cm"),
        df_products.product_width_cm.alias("product_width_cm"),
        df_products.product_height_cm.alias("product_height_cm")
    )


df_products_dimension=df_products_dimension.fillna({
    "product_category_name_english": "UNKNOWN",
    "product_name_length": 0,
    "product_description_length": 0,
    "product_photos_qty": 0,
    "product_weight_g": 0,
    "product_length_cm": 0,
    "product_width_cm": 0,
    "product_height_cm": 0})

In [0]:
df_review_dimension= df_order_reviews.select(
    df_order_reviews.review_id.alias("review_id"),
    df_order_reviews.review_comment_title.alias("review_title"),
    df_order_reviews.review_comment_message.alias("review_comment_message")
)

In [0]:
from pyspark.sql.window import Window

df_order_status_dimension= df_orders.select(
    df_orders.order_status.alias("order_status")
    ).distinct()

df_order_status_dimension=df_order_status_dimension.withColumn("order_status_id", sf.row_number().over(Window.orderBy(df_order_status_dimension.order_status)))

df_order_status_dimension=df_order_status_dimension.select("order_status_id","order_status")

In [0]:
df_payments_type_dimension= df_order_payments.select("payment_type").distinct()

df_payments_type_dimension=df_payments_type_dimension.withColumn("payment_type_id", sf.row_number().over(Window.orderBy(df_payments_type_dimension.payment_type)))

df_payments_type_dimension=df_payments_type_dimension.select("payment_type_id","payment_type")

In [0]:
df_all_dates= df_order_reviews.select(sf.col("review_creation_date").alias("date_timestamp")
    ).union(df_order_reviews.select(sf.col("review_answer_timestamp").alias("date_timestamp"))
    ).union(df_orders.select(sf.col("order_purchase_timestamp").alias("date_timestamp"))
    ).union(df_order_items.select(sf.col("shipping_limit_date").alias("date_timestamp"))
    ).union(df_orders.select(sf.col("order_approved_at").alias("date_timestamp"))
    ).union(df_orders.select(sf.col("order_delivered_carrier_date").alias("date_timestamp"))
    ).union(df_orders.select(sf.col("order_delivered_customer_date").alias("date_timestamp"))
    ).union(df_orders.select(sf.col("order_estimated_delivery_date").alias("date_timestamp"))
    ).distinct()


df_all_dates=df_all_dates.dropna()
df_all_dates= df_all_dates.distinct()

df_all_dates=df_all_dates.withColumn("date_id", sf.row_number().over(Window.orderBy(df_all_dates.date_timestamp)))

df_date_dimension=df_all_dates.select(
    sf.col("date_id"),
    sf.col("date_timestamp"),
    sf.year(sf.col("date_timestamp")).alias("date_year"),
    sf.month(sf.col("date_timestamp")).alias("date_month"),
    sf.date_format(sf.col("date_timestamp"),"MMM").alias("date_month_name"),
    sf.dayofmonth(sf.col("date_timestamp")).alias("date_day_of_month"),
    sf.dayofweek(sf.col("date_timestamp")).alias("day_of_week"),
    sf.when(sf.dayofweek(sf.col("date_timestamp"))<6,"weekday").otherwise("weekend").alias("date_day_type")
)


In [0]:
df_orders_fact=df_orders.join(
    df_customers, how="inner", on="customer_id").join(
    df_order_status_dimension, how="left", on="order_status").select(
        "order_id",
        "customer_unique_id",
        "order_status_id",
        sf.col("order_purchase_timestamp").alias("order_purchase_date"),
        sf.col("order_approved_at").alias("order_approved_date"),
        sf.col("order_delivered_carrier_date").alias("order_delivered_carrier_date"),
        sf.col("order_delivered_customer_date").alias("order_delivered_customer_date"),
        sf.col("order_estimated_delivery_date").alias("order_delivered_estimate_date")
        )
    
df_orders_fact=df_orders_fact.join(df_date_dimension, how="left", on=df_date_dimension.date_timestamp==df_orders_fact.order_purchase_date).select(
    "order_id",
    "customer_unique_id",
    "order_status_id",
    "order_approved_date",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_delivered_estimate_date",
    sf.col("date_id").alias("order_purchase_date_fk"))


df_orders_fact=df_orders_fact.join(df_date_dimension, how="left", on=df_date_dimension.date_timestamp==df_orders_fact.order_approved_date).select(
    "order_id",
    "customer_unique_id",
    "order_status_id",
    "order_purchase_date_fk",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_delivered_estimate_date",
    sf.col("date_id").alias("order_approved_date_fk"))


df_orders_fact=df_orders_fact.join(df_date_dimension, how="left", on=df_date_dimension.date_timestamp==df_orders_fact.order_delivered_carrier_date).select(
    "order_id",
    "customer_unique_id",
    "order_status_id",
    "order_purchase_date_fk",
    "order_approved_date_fk",
    "order_delivered_customer_date",
    "order_delivered_estimate_date",
    sf.col("date_id").alias("order_delivered_carrier_date_fk"))


df_orders_fact=df_orders_fact.join(df_date_dimension, how="left", on=df_date_dimension.date_timestamp==df_orders_fact.order_delivered_customer_date).select(
    "order_id",
    "customer_unique_id",
    "order_status_id",
    "order_purchase_date_fk",
    "order_approved_date_fk",
    "order_delivered_carrier_date_fk",
    "order_delivered_customer_date",
    "order_delivered_estimate_date",
    sf.col("date_id").alias("order_delivered_customer_date_fk"))


df_orders_fact=df_orders_fact.join(df_date_dimension, how="left", on=df_date_dimension.date_timestamp==df_orders_fact.order_delivered_estimate_date).select(
    sf.col("order_id").alias("order_id_pk"),
    sf.col("customer_unique_id").alias("customer_unique_id_fk"),
    sf.col("order_status_id").alias("order_status_id_fk"),
    "order_purchase_date_fk",
    "order_approved_date_fk",
    "order_delivered_carrier_date_fk",
    "order_delivered_customer_date_fk",
    sf.col("date_id").alias("order_delivered_estimate_date_fk"))

In [0]:
df_review_fact=df_order_reviews.join(
    df_orders_fact,how="inner",on=df_orders_fact.order_id_pk==df_order_reviews.order_id
).select(
    sf.col("review_id").alias("review_id_fk"),
    sf.col("order_id_pk").alias("order_id_fk"),
    "review_score",
    "review_creation_date",
    "review_answer_timestamp"
    )

df_review_fact=df_review_fact.join(
    df_date_dimension,how="left",on=df_date_dimension.date_timestamp==df_order_reviews.review_creation_date).select(
        "review_id_fk",
        sf.col("order_id_fk"),
        "review_score",
        "review_answer_timestamp",
        sf.col("date_id").alias("review_created_at_fk"))
    
df_review_fact=df_review_fact.join(
    df_date_dimension,how="left",on=df_date_dimension.date_timestamp==df_review_fact.review_answer_timestamp).select(
        "review_id_fk",
        "order_id_fk",
        "review_score",
        "review_created_at_fk",
        sf.col("date_id").alias("review_answered_at_fk"))

In [0]:
df_payments_fact=df_order_payments.join(df_orders_fact, how="inner", on=df_orders_fact.order_id_pk==df_order_payments.order_id).select(
    sf.col("order_id").alias("order_id_fk"),
    sf.col("payment_sequential"),
    sf.col("payment_type").alias("payment_type"),
    sf.col("payment_installments").alias("payment_installments"),
    sf.col("payment_value").alias("payment_value")
)


df_payments_fact= df_payments_fact.join(df_payments_type_dimension, how="left", on="payment_type").select(
    "order_id_fk",
    sf.col("payment_sequential").alias("payment_sequential"),
    sf.col("payment_type_id").alias("payment_type_id_fk"),
    sf.col("payment_value").alias("payment_value"),
    sf.col("payment_installments").alias("payment_installments")
    )

In [0]:
df_order_items_fact=df_order_items.join(
    df_products_dimension, how="inner", on=df_products_dimension.product_id==df_order_items.product_id
    ).join(
    df_orders_fact, how="inner", on=df_orders_fact.order_id_pk==df_order_items.order_id
    ).join(
        df_sellers_dimension, how="inner", on=df_sellers_dimension.seller_id==df_order_items.seller_id
    ).select(
        "order_id_pk",
        "order_item_id",
        df_order_items.product_id,
        df_order_items.seller_id,
        "shipping_limit_date",
        "price",
        "freight_value"
    )

df_order_items_fact= df_order_items_fact.join(df_date_dimension, how="left", on=df_date_dimension.date_timestamp==df_order_items.shipping_limit_date).select(
    sf.col("order_id_pk").alias("order_id_fk"),
    sf.col("order_item_id").alias("order_item_id"),
    sf.col("product_id").alias("product_id_fk"),
    sf.col("seller_id").alias("seller_id_fk"),
    sf.col("date_id").alias("shipping_limit_date_fk"),
    sf.col("price").alias("price"),
    sf.col("freight_value").alias("freight_value")
    )

In [0]:
df_customers_dimension=df_customers_dimension.withColumnRenamed("customer_unique_id", "customer_unique_id_pk")
df_sellers_dimension=df_sellers_dimension.withColumnRenamed("seller_id", "seller_id_pk")
df_products_dimension=df_products_dimension.withColumnRenamed("product_id", "product_id_pk")
df_order_status_dimension=df_order_status_dimension.withColumnRenamed("order_status_id", "order_status_id_pk")
df_payments_type_dimension=df_payments_type_dimension.withColumnRenamed("payment_type_id", "payment_type_id_pk")
df_date_dimension=df_date_dimension.withColumnRenamed("date_id", "date_id_pk")
df_review_dimension=df_review_dimension.withColumnRenamed("review_id", "review_id_pk")


In [0]:
df_customers_dimension.write.mode("overwrite").parquet(customer_dimension_write_path)
df_sellers_dimension.write.mode("overwrite").parquet(seller_dimension_write_path)
df_products_dimension.write.mode("overwrite").parquet(product_dimension_write_path)
df_order_status_dimension.write.mode("overwrite").parquet(order_status_dimension_write_path)
df_payments_type_dimension.write.mode("overwrite").parquet(payment_type_dimension_write_path)
df_date_dimension.write.mode("overwrite").parquet(date_dimension_write_path)
df_review_dimension.write.mode("overwrite").parquet(review_dimension_write_path)
df_orders_fact.write.mode("overwrite").parquet(order_fact_write_path)
df_review_fact.write.mode("overwrite").parquet(review_fact_write_path)
df_payments_fact.write.mode("overwrite").parquet(payment_fact_write_path)
df_order_items_fact.write.mode("overwrite").parquet(order_item_fact_write_path)